# LC-KSVD2 sparse-code SVM: top-10-discriminative-atoms per class

Trains a Linear SVM (`LinearSVC`, same `MaxAbsScaler` + `GridSearchCV` setup used by
`lc_ksvd/inference/train.py::train_svm` and the raw-patch baseline notebook) on a
reduced feature set: instead of all 2592 dictionary atoms in `Gamma`, only the union
of the top-10 highest-firing-rate atoms per class (60 atom indices total, deduped) is
used as the feature space.

Data:
- Train: `outputs/sparse_codes/previous/train_sparse_codes_lcksvd2.npz` (`Gamma` 2592 x 204935, 6 classes)
- Test:  `outputs/sparse_codes/previous/test_sparse_codes_lcksvd2.npz` (`Gamma` 2592 x 80954, 6 classes)

In [1]:
import logging

import joblib
import numpy as np
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MaxAbsScaler
from sklearn.svm import LinearSVC

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s")

RANDOM_SEED = 42

SPARSE_CODE_DIR = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/previous"
TRAIN_SPARSE_CODES_PATH = f"{SPARSE_CODE_DIR}/train_sparse_codes_lcksvd2.npz"
TEST_SPARSE_CODES_PATH = f"{SPARSE_CODE_DIR}/test_sparse_codes_lcksvd2.npz"

MODEL_PATH = "/home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/models/lcksvd2_top_atoms_svm_model.pkl"

In [2]:
# Top-10 atoms per class, by firing rate
TOP_ATOMS_PER_CLASS = {
    0: [1003, 1979, 1230, 1040, 330, 91, 342, 205, 967, 565],
    1: [2037, 969, 2454, 1657, 2349, 2173, 360, 407, 1815, 1666],
    2: [675, 2546, 1958, 2056, 842, 120, 2305, 1522, 720, 159],
    3: [2341, 1267, 1201, 157, 2419, 2540, 1273, 2401, 974, 1885],
    4: [2275, 2156, 1297, 2207, 2385, 753, 2511, 2525, 2353, 340],
    5: [2321, 863, 1449, 2168, 2162, 1744, 1295, 2451, 1289, 2183],
}

# Union of all selected atom indices (sorted, deduped) -- the reduced feature space
SELECTED_ATOMS = sorted({atom for atoms in TOP_ATOMS_PER_CLASS.values() for atom in atoms})
print(f"{len(SELECTED_ATOMS)} unique atoms selected out of "
      f"{sum(len(v) for v in TOP_ATOMS_PER_CLASS.values())} total (per-class lists overlap if this is < 60)")

60 unique atoms selected out of 60 total (per-class lists overlap if this is < 60)


In [3]:
def load_selected_gamma(path, selected_atoms):
    data = np.load(path)
    Gamma, labels = data["Gamma"], data["labels"]
    logger.info(f"Loaded {path}: Gamma {Gamma.shape}, labels {labels.shape}")
    X = Gamma[selected_atoms, :].T  # (n_samples, n_selected_atoms)
    return X, labels


X_train, y_train = load_selected_gamma(TRAIN_SPARSE_CODES_PATH, SELECTED_ATOMS)
X_test, y_test = load_selected_gamma(TEST_SPARSE_CODES_PATH, SELECTED_ATOMS)

print("X_train:", X_train.shape, "X_test:", X_test.shape)
print("train class counts:", dict(zip(*np.unique(y_train, return_counts=True))))
print("test class counts:", dict(zip(*np.unique(y_test, return_counts=True))))

INFO Loaded /home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/previous/train_sparse_codes_lcksvd2.npz: Gamma (2592, 204935), labels (204935,)


INFO Loaded /home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/sparse_codes/previous/test_sparse_codes_lcksvd2.npz: Gamma (2592, 80954), labels (80954,)


X_train: (204935, 60) X_test: (80954, 60)
train class counts: {np.int64(0): np.int64(54909), np.int64(1): np.int64(23440), np.int64(2): np.int64(48951), np.int64(3): np.int64(34403), np.int64(4): np.int64(36906), np.int64(5): np.int64(6326)}
test class counts: {np.int64(0): np.int64(52905), np.int64(1): np.int64(2779), np.int64(2): np.int64(14300), np.int64(3): np.int64(5369), np.int64(4): np.int64(5476), np.int64(5): np.int64(125)}


## Train: Linear SVM (LinearSVC) on top-atom features

In [4]:
def train_svm(X, y, cv_splits=5):
    pipeline = Pipeline([
        ("scaler", MaxAbsScaler()),
        ("svm", LinearSVC(dual=False, max_iter=10000, class_weight="balanced", random_state=RANDOM_SEED)),
    ])
    param_grid = {"svm__C": [0.1, 1.0, 10.0, 100.0]}
    grid = GridSearchCV(pipeline, param_grid, cv=cv_splits, scoring="f1_macro", n_jobs=-1, verbose=2)
    grid.fit(X, y)
    print("\nBest parameters:", grid.best_params_)
    print("Best CV score (f1_macro):", grid.best_score_)
    return grid.best_estimator_


svm_clf = train_svm(X_train, y_train)
joblib.dump(svm_clf, MODEL_PATH)
logger.info(f"Saved SVM model -> {MODEL_PATH}")

Fitting 5 folds for each of 4 candidates, totalling 20 fits


INFO Saved SVM model -> /home/chest_ct/code/models/lc-ksvd/src/lc_ksvd/outputs/models/lcksvd2_top_atoms_svm_model.pkl



Best parameters: {'svm__C': 100.0}
Best CV score (f1_macro): 0.16669104602915166


## Evaluate on test set

In [5]:
y_pred = svm_clf.predict(X_test)

print("=== LC-KSVD2 top-atoms LinearSVC ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nPer-class accuracy (one-vs-rest):")
for cls in sorted(TOP_ATOMS_PER_CLASS):
    class_acc = ((y_test == cls) == (y_pred == cls)).mean()
    print(f"  class {cls}: {class_acc:.4f}")

=== LC-KSVD2 top-atoms LinearSVC ===
Accuracy: 0.2109

Classification Report:
              precision    recall  f1-score   support

           0       0.77      0.11      0.19     52905
           1       0.18      0.01      0.02      2779
           2       0.20      0.44      0.28     14300
           3       0.12      0.91      0.21      5369
           4       0.25      0.02      0.04      5476
           5       0.00      0.01      0.00       125

    accuracy                           0.21     80954
   macro avg       0.25      0.25      0.12     80954
weighted avg       0.57      0.21      0.19     80954


Confusion Matrix:
[[ 5798    62 22939 23075   198   833]
 [   54    31   337  2324    29     4]
 [ 1586    12  6235  6227    62   178]
 [   49    35   247  4885    99    54]
 [   48    30   823  4423   127    25]
 [    0     0     3   121     0     1]]

Per-class accuracy (one-vs-rest):
  class 0: 0.3966
  class 1: 0.9643
  class 2: 0.5996
  class 3: 0.5472
  class 4: 0.9291
